# 3 · Generar las fotos de Lía (SDXL + su LoRA), estilo foto de móvil
Usa el LoRA de `lia2/lora`. Genera las publicaciones del plan de 30 días con estilo de selfie / foto casual y las guarda en
`lia2/salida_iphone` junto a `captions.csv` (texto con el aviso de IA). Cambia `DESDE` y `HASTA` para hacerlo por tandas.
Las fotos de espejo piden el iPhone con su logo visible y el móvil tapando la cara (la IA no siempre lo dibuja bien: se descartan las que salgan mal).
**GPU T4.** Aproximadamente 1–2 min por imagen con la pasada de detalle (estimado, sin medir; ponla en `DETALLE = False` para ir más rápido). **Revisa cada imagen a mano** (manos, cara, proporciones, texto raro)
y publica siempre con la etiqueta de IA.

In [ ]:
import os, glob
if os.path.exists('/kaggle/working'):           # Kaggle (alternativa gratis cuando Colab no da GPU)
    PLATAFORMA = 'kaggle'
    BASE = '/kaggle/working/lia2'
else:                                           # Colab: todo se guarda en tu Google Drive
    PLATAFORMA = 'colab'
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/lia2'
for d in ('candidatas', 'dataset', 'lora'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
import torch; print(PLATAFORMA, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA -> activa la GPU en los ajustes del cuaderno')

In [ ]:
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0 transformers accelerate peft safetensors compel

In [ ]:
STYLE = 'amateur iPhone photo, candid, unedited, slight grain, imperfect framing'
NEGATIVE = 'nsfw, nudity, see-through clothing, nipples, topless, freckles, deformed, bad anatomy, deformed iris, uncanny, bad teeth, extra fingers, blurry, watermark, plastic skin, exaggerated proportions, huge breasts, disproportionate body, studio lighting, professional photography, bokeh, retouched, airbrushed, HDR, cinematic, deformed phone, phone without camera lenses'
OUT = 'salida_iphone'
PLAN = [
 {
  "pillar": "playa_sol",
  "prompt": "photo of lia_character woman, paddle board at sunrise, athletic swimwear, confident smile",
  "caption": "Paddle board at sunrise, athletic swimwear, confident smile ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "gym",
  "prompt": "photo of lia_character woman, resting between sets on a bench, towel around neck, gym lighting",
  "caption": "Resting between sets on a bench, towel around neck, gym lighting ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "playa_sol",
  "prompt": "photo of lia_character woman, lounging by a rooftop pool in a one-piece swimsuit and sunglasses",
  "caption": "Lounging by a rooftop pool in a one-piece swimsuit and sunglasses ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, sitting on an outdoor cafe sofa holding an iced coffee, blue varsity jacket, ripped jeans, white sneakers, hand covering part of her face",
  "caption": "Sitting on an outdoor cafe sofa holding an iced coffee, blue varsity jacket, ripped jeans, white sneakers, hand covering part of her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "sin_cara",
  "prompt": "photo of lia_character woman, torso and waist in a cropped top and low-rise jeans at golden hour, face out of frame",
  "caption": "Torso and waist in a cropped top and low-rise jeans at golden hour, face out of frame ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "lifestyle",
  "prompt": "photo of lia_character woman, sitting in a sunny café in a fitted scoop-neck top and high-waist jeans with iced coffee",
  "caption": "Sitting in a sunny café in a fitted scoop-neck top and high-waist jeans with iced coffee ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt",
  "caption": "Mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, seen from behind in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, fitted black crop top and floral skirt, holding phone up to the mirror",
  "caption": "Seen from behind in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, fitted black crop top and floral skirt, holding phone up to the mirror ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt",
  "caption": "Mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "sin_cara",
  "prompt": "photo of lia_character woman, torso and waist in a cropped top and low-rise jeans at golden hour, face out of frame",
  "caption": "Torso and waist in a cropped top and low-rise jeans at golden hour, face out of frame ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "salida_noche",
  "prompt": "photo of lia_character woman, in a fitted black mini dress at a rooftop bar, city lights, playful look at camera",
  "caption": "In a fitted black mini dress at a rooftop bar, city lights, playful look at camera ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in bedroom with a white door, light wood floor and grey bedspread, tall mirror with a thin black frame, frame edges visible, white fitted crop top and light blue jeans, phone covering half her face",
  "caption": "Mirror photo in bedroom with a white door, light wood floor and grey bedspread, tall mirror with a thin black frame, frame edges visible, white fitted crop top and light blue jeans, phone covering half her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt",
  "caption": "Mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "gym",
  "prompt": "photo of lia_character woman, mirror photo in a modern gym, matching sports bra and leggings, holding phone",
  "caption": "Mirror photo in a modern gym, matching sports bra and leggings, holding phone ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "sin_cara",
  "prompt": "photo of lia_character woman, legs and sandals by a rooftop pool in a summer swimsuit, bright sun",
  "caption": "Legs and sandals by a rooftop pool in a summer swimsuit, bright sun ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "gym",
  "prompt": "photo of lia_character woman, mirror photo in a gym changing area after a workout, cropped zip hoodie and biker shorts, holding phone",
  "caption": "Mirror photo in a gym changing area after a workout, cropped zip hoodie and biker shorts, holding phone ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "sin_cara",
  "prompt": "photo of lia_character woman, back view standing at a gym mirror in a sports bra and high-waist leggings, athletic toned back",
  "caption": "Back view standing at a gym mirror in a sports bra and high-waist leggings, athletic toned back ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt",
  "caption": "Mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, harsh ceiling lights, phone covering her face, fitted black scoop-neck top, modest cleavage, floral skirt ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face",
  "caption": "Mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, sitting on an outdoor cafe sofa holding an iced coffee, blue varsity jacket, ripped jeans, white sneakers, hand covering part of her face",
  "caption": "Sitting on an outdoor cafe sofa holding an iced coffee, blue varsity jacket, ripped jeans, white sneakers, hand covering part of her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face",
  "caption": "Mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face",
  "caption": "Mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "sin_cara",
  "prompt": "photo of lia_character woman, back view standing at a gym mirror in a sports bra and high-waist leggings, athletic toned back",
  "caption": "Back view standing at a gym mirror in a sports bra and high-waist leggings, athletic toned back ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, at night, black hoodie and crop top, phone covering her face, grainy",
  "caption": "Mirror photo in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, at night, black hoodie and crop top, phone covering her face, grainy ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "gym",
  "prompt": "photo of lia_character woman, mirror photo in a gym changing area after a workout, cropped zip hoodie and biker shorts, holding phone",
  "caption": "Mirror photo in a gym changing area after a workout, cropped zip hoodie and biker shorts, holding phone ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in bedroom with a white door, light wood floor and grey bedspread, tall mirror with a thin black frame, frame edges visible, white fitted crop top and light blue jeans, phone covering half her face",
  "caption": "Mirror photo in bedroom with a white door, light wood floor and grey bedspread, tall mirror with a thin black frame, frame edges visible, white fitted crop top and light blue jeans, phone covering half her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "lifestyle",
  "prompt": "photo of lia_character woman, on a city street in oversized blazer and bike shorts, golden hour",
  "caption": "On a city street in oversized blazer and bike shorts, golden hour ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "playa_sol",
  "prompt": "photo of lia_character woman, paddle board at sunrise, athletic swimwear, confident smile",
  "caption": "Paddle board at sunrise, athletic swimwear, confident smile ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, seen from behind in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, fitted black crop top and floral skirt, holding phone up to the mirror",
  "caption": "Seen from behind in small white-tiled bathroom, rectangular mirror with a thin black frame over a white sink, grey towel on a hook, frame edges visible, fitted black crop top and floral skirt, holding phone up to the mirror ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 },
 {
  "pillar": "selfie_espejo",
  "prompt": "photo of lia_character woman, mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face",
  "caption": "Mirror photo in bedroom with a white door, light wood floor and grey bedspread, white fitted tank top and high-waist jeans, phone covering half her face ✨\n\n✨ Imagen generada con IA · Virtual creator\n\n#aiinfluencer #virtualcreator #aigenerated #fitnessinspo #lifestyle"
 }
]
DESDE, HASTA = 1, 10     # días del plan a generar
DETALLE = True          # pasada de detalle: amplía la foto y repasa cara y piel (más definición, +1 min por foto)
ESCALA_DETALLE, FUERZA_DETALLE = 1.25, 0.3

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
pipe = StableDiffusionXLPipeline.from_pretrained('SG161222/RealVisXL_V4.0', torch_dtype=torch.float16,
                                                 variant='fp16', use_safetensors=True).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)

from compel import Compel, ReturnedEmbeddingsType
compel = Compel(tokenizer=[pipe.tokenizer, pipe.tokenizer_2], text_encoder=[pipe.text_encoder, pipe.text_encoder_2],
                returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED,
                requires_pooled=[False, True], truncate_long_prompts=False)
def codificar(prompt, negative):
    # los guiones se quitan porque compel los usaría como símbolo de peso (p. ej. "white-tiled")
    cond, pooled = compel(prompt.replace('-', ' ')); ncond, npooled = compel(negative.replace('-', ' '))
    cond, ncond = compel.pad_conditioning_tensors_to_same_length([cond, ncond])
    return dict(prompt_embeds=cond, pooled_prompt_embeds=pooled, negative_prompt_embeds=ncond, negative_pooled_prompt_embeds=npooled)

lora_dir = f'{BASE}/lora'
if not os.path.exists(f'{lora_dir}/pytorch_lora_weights.safetensors'):   # Kaggle: viene como Input (salida del cuaderno 2)
    hallado = sorted(glob.glob('/kaggle/input/**/pytorch_lora_weights.safetensors', recursive=True), key=lambda p: ('checkpoint' in p, len(p)))   # el final, no los checkpoint intermedios
    assert hallado, 'No encuentro el LoRA: en Kaggle añade la salida del cuaderno 2 con Add Data > Notebook Output'
    lora_dir = os.path.dirname(hallado[0])
pipe.load_lora_weights(lora_dir, weight_name='pytorch_lora_weights.safetensors')
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image
refinar = StableDiffusionXLImg2ImgPipeline(**pipe.components)   # comparte modelo y LoRA: no gasta más memoria
pipe.enable_vae_tiling(); refinar.enable_vae_tiling()
import csv, os, random
os.makedirs(f'{BASE}/{OUT}', exist_ok=True)
cap = f'{BASE}/{OUT}/captions.csv'
if not os.path.exists(cap):
    open(cap, 'w', encoding='utf-8').write('dia,pilar,archivo,caption\n')
VARIANTES_ESPEJO = 4   # selfies de espejo: se generan varias versiones para quedarte con la del móvil bien hecho
for n in range(DESDE, HASTA + 1):
  it = PLAN[n - 1]
  for v in range(VARIANTES_ESPEJO if 'mirror photo' in it['prompt'] else 1):
    f = f'{BASE}/{OUT}/{n:02d}_{it["pillar"]}' + (f'_v{v + 1}' if 'mirror photo' in it['prompt'] else '') + '.png'
    if os.path.exists(f):
        continue
    seed = random.randint(0, 2**31)
    g = torch.Generator('cuda').manual_seed(seed)
    if 'mirror photo' in it['prompt']:
        cam = 'mirror photo, iPhone 16 Pro in a black case with three camera lenses and the Apple logo on its back'
    else:
        cam = 'shot on a phone, casual framing'
    emb = codificar(f'{STYLE}, {cam}, {it["prompt"]}', NEGATIVE)
    img = pipe(**emb, width=832, height=1216, num_inference_steps=30, guidance_scale=4.5, generator=g,
               cross_attention_kwargs={'scale': 0.9}).images[0]
    if DETALLE:
        try:
            w2, h2 = int(832 * ESCALA_DETALLE) // 8 * 8, int(1216 * ESCALA_DETALLE) // 8 * 8
            img = refinar(**emb, image=img.resize((w2, h2), Image.LANCZOS), strength=FUERZA_DETALLE, num_inference_steps=30,
                          guidance_scale=4.5, generator=g, cross_attention_kwargs={'scale': 0.9}).images[0]
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); print('sin memoria para la pasada de detalle: se guarda la foto sin repasar')
    img.save(f)
    with open(cap, 'a', newline='', encoding='utf-8') as fh:
        csv.writer(fh).writerow([n, it['pillar'], os.path.basename(f), it['caption']])
    print(n, it['pillar'], 'seed', seed, flush=True)

## Efecto «foto de móvil» (opcional)
Copia cada imagen de `salida_iphone` a `salida_iphone/movil/*.jpg` con grano de sensor, un poco menos de nitidez, viñeteo suave y compresión JPEG, como una foto subida desde el móvil. Solo cambia la estética: **no** añade metadatos falsos de cámara. Las originales se conservan.

In [ ]:
import glob, numpy as np
from PIL import Image, ImageFilter, ImageEnhance
os.makedirs(f'{BASE}/{OUT}/movil', exist_ok=True)
def efecto_movil(img, seed):
    rng = np.random.default_rng(seed)
    im = img.convert('RGB').filter(ImageFilter.GaussianBlur(0.6))
    a = np.asarray(im).astype(np.float32) + rng.normal(0, 3.5, (im.size[1], im.size[0], 3))
    h, w = a.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]
    a *= (1 - 0.10 * (((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2))[..., None]
    return ImageEnhance.Contrast(Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))).enhance(1.05)
for f in sorted(glob.glob(f'{BASE}/{OUT}/*.png')):
    out = f'{BASE}/{OUT}/movil/' + os.path.basename(f)[:-4] + '.jpg'
    if not os.path.exists(out):
        efecto_movil(Image.open(f), abs(hash(f)) % 2**32).save(out, quality=82)
print('Listo: lia2/' + OUT + '/movil')